# MOEX Portfolio Optimizer

Автоматический подбор оптимального инвестиционного портфеля акций MOEX.

## Пайплайн
1. Загрузка данных с MOEX ISS API
2. Фильтрация по ликвидности и аномалиям
3. Расчёт матрицы корреляций
4. Построение графа корреляций и поиск максимальной клики
5. Markowitz Mean-Variance оптимизация
6. Визуализация результатов

In [ ]:
import sys
from pathlib import Path

# Добавляем src в путь для импорта модулей
sys.path.insert(0, str(Path("..") / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from moex_portfolio.config import (
    CORR_THRESHOLD, MIN_TURNOVER, MIN_OBSERVATIONS,
    START_DATE, END_DATE, FIGURES_DIR
)
from moex_portfolio.data_loader import load_all_data
from moex_portfolio.filters import prepare_returns
from moex_portfolio.correlation import compute_correlation_matrix, plot_correlation_heatmap
from moex_portfolio.graph_analysis import build_correlation_graph, find_max_clique
from moex_portfolio.visualization import (
    plot_full_graph, plot_clique_on_graph, plot_clique_separate,
    plot_clique_heatmap, plot_total_returns
)
from moex_portfolio.metrics import portfolio_metrics
from moex_portfolio.optimizer import max_sharpe_portfolio, min_variance_portfolio, efficient_frontier

print(f"Период данных: {START_DATE} — {END_DATE}")
print(f"Порог корреляции: {CORR_THRESHOLD}")
print(f"Минимальный оборот: {MIN_TURNOVER / 1_000_000:.0f}M RUB")

## 1. Загрузка данных

In [ ]:
raw_data = load_all_data(use_cache=True)
print(f"Загружено: {raw_data.shape[0]} дней, {raw_data.shape[1]} столбцов")

## 2. Фильтрация и расчёт доходностей

In [ ]:
returns, valid_tickers = prepare_returns(raw_data)
print(f"После фильтрации: {len(valid_tickers)} акций, {len(returns)} периодов")
returns.head()

## 3. Матрица корреляций

In [ ]:
corr = compute_correlation_matrix(returns)
fig = plot_correlation_heatmap(corr, title="Матрица корреляций акций MOEX")
plt.show()

## 4. Граф корреляций и максимальная клика

In [ ]:
G = build_correlation_graph(corr, threshold=CORR_THRESHOLD)
clique = find_max_clique(G)
print(f"Максимальная клика: {len(clique)} акций")
print(f"Акции: {', '.join(clique)}")

In [ ]:
fig = plot_full_graph(G, clique=clique)
plt.show()

In [ ]:
fig = plot_clique_separate(G, clique)
plt.show()

In [ ]:
fig = plot_clique_heatmap(returns, clique)
plt.show()

## 5. Доходности акций клики

In [ ]:
fig = plot_total_returns(returns, clique)
plt.show()

## 6. Оптимизация портфеля (Markowitz)

In [ ]:
clique_returns = returns[clique]
mean_ret = clique_returns.mean()
cov = clique_returns.cov()

# Максимальный Sharpe ratio
opt = max_sharpe_portfolio(mean_ret, cov)
print("=== Максимальный Sharpe Ratio ===")
print(f"Доходность: {opt['return']:.2%}")
print(f"Волатильность: {opt['volatility']:.2%}")
print(f"Sharpe: {opt['sharpe']:.3f}")
print()
for t, w in sorted(zip(clique, opt['weights']), key=lambda x: -x[1]):
    print(f"  {t}: {w:.2%}")

In [ ]:
# Минимальная волатильность
min_var = min_variance_portfolio(mean_ret, cov)
print("=== Минимальная волатильность ===")
print(f"Доходность: {min_var['return']:.2%}")
print(f"Волатильность: {min_var['volatility']:.2%}")
print(f"Sharpe: {min_var['sharpe']:.3f}")
print()
for t, w in sorted(zip(clique, min_var['weights']), key=lambda x: -x[1]):
    print(f"  {t}: {w:.2%}")

## 7. Эффективный фронтер

In [ ]:
ef = efficient_frontier(mean_ret, cov, n_points=50)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(ef['volatility'], ef['return'], c=ef['sharpe'], cmap='viridis', s=10)
ax.scatter(opt['volatility'], opt['return'], marker='*', s=300, c='red', label='Max Sharpe', zorder=5)
ax.scatter(min_var['volatility'], min_var['return'], marker='*', s=300, c='blue', label='Min Variance', zorder=5)
ax.set_xlabel('Волатильность (годовая)')
ax.set_ylabel('Доходность (годовая)')
ax.set_title('Эффективный фронтер')
ax.legend()
plt.colorbar(ax.collections[0], ax=ax, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 8. Сводные метрики

In [ ]:
metrics = portfolio_metrics(
    opt['weights'], mean_ret, cov, returns=clique_returns
)
print("=== Метрики оптимального портфеля ===")
print(f"Годовая доходность:      {metrics['return']:.2%}")
print(f"Годовая волатильность:   {metrics['volatility']:.2%}")
print(f"Sharpe Ratio:            {metrics['sharpe']:.3f}")
print(f"Sortino Ratio:           {metrics['sortino']:.3f}")
print(f"Максимальная просадка:   {metrics['max_drawdown']:.2%}")